# Performance em GPU [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chcomin/curso-visao-computacional-2025/blob/main/M13_desempenho_e_eficiencia/2%20-%20Abordagens%20para%20melhorar%20a%20performance%20(GPU).ipynb)

Veremos algumas estratégias para melhorar a perfomance em GPUs

### Precisão numérica

GPUs modernas realizam operações de forma muito mais eficiente em float16. Mas operações em float16 possuem menor precisão, então é preciso tomar cuidado com a estabilidade numérica dos resultados. Existem algumas abordagens para realizar operações em menor precisão e ao mesmo tempo evitar erros numéricos. 

Compararemos as seguintes situações que possuem diferentes relações custo x precisão:

1. Operações em float64, o que dá o resultado mais preciso possível, mas é menos eficiente
2. Operações em float32, que é o padrão do Pytorch
3. O Pytorch disponibiliza a chamada *automatic mixed precision*. Quando é detectado que uma operação pode ser realizada sem muita perda de precisão, o Pytorch automaticamente realiza a operação em menor precisão. Para isso, é usado o contexto `torch.autocast`
4. Realizar operações em float16, o que é extremamente eficiente em GPUs modernas. Mas é preciso tomar cuidado, por exemplo, não é recomendado realizar o backpropagation em float16
5. Em GPUs recentes o Pytorch possui a função `torch.set_float32_matmul_precision`, que permite o uso de um tipo especial de dado, o chamado *tensorfloat32*. Esse tipo de dado é usado exclusivamente por tensor cores. Ele permite uma precisão próxima de float32 mas com a eficiência de float16

Aplicaremos essas técnicas em multiplicações matriciais

In [1]:
import torch


class PerfRecorder:
    """Registra o tempo de execução na GPU e o uso de memória."""

    def __init__(self):

        self.gpu_start = torch.cuda.Event(enable_timing=True)
        self.gpu_end = torch.cuda.Event(enable_timing=True)  

    def start(self):
        """Inicia registro."""

        torch.cuda.reset_peak_memory_stats()
        self.gpu_start.record() 

    def end(self):
        """Encerra registro."""

        self.gpu_end.record()
        torch.cuda.synchronize()

        # Tempo de execução na GPU, em ms
        t_gpu = self.gpu_start.elapsed_time(self.gpu_end)
        # Uso de memória em GiB
        max_memory = torch.cuda.max_memory_allocated()/2**30
    
        return t_gpu, max_memory
    
def benchmark(shape, dtype, n=5, n_warm=2):
    """Realiza `n` multiplicações matriciais entre matrizes de tamanho
    shape[0]xshape[1] x shape[1]xshape[0]. Retorna o tempo médio de cada 
    multiplicação, a memória utilizada e a média dos valores do resultado."""

    recorder = PerfRecorder()

    nr, nc = shape

    torch.manual_seed(0)
    x1 = torch.randn(nr, nc, dtype=dtype, device="cuda")
    x2 = torch.randn(nc, nr, dtype=dtype, device="cuda")

    for _ in range(n_warm):
        _ = torch.matmul(x1, x2)

    recorder.start()
    for _ in range(n):
        r = torch.matmul(x1, x2)
    t_gpu, max_memory = recorder.end()
    t_gpu /= n

    # Média dos valores do resultado do cálculo
    mean_val = r.abs().mean().item()
    
    return t_gpu, max_memory, mean_val

# Memória disponível na GPU em GiB
mem_size = 12
# Quantidade de valores que podem ser alocados em float64, excluindo 2 GiB para
# evitar problemas de memória
nv = (mem_size-2)*2**30//8
# //2 porque vamos alocar duas matrizes
nv = nv//2
# Tamanho das matrizes. As dimensões terem ao menos tamanho 512 garante que
# a GPU será utilizada ao máximo
mat_shape = (512, nv//512)

In [2]:
# float64
t_f64, m_f64, v_f64 = benchmark(mat_shape, torch.float64)
# float32
t_f32, m_f32, v_f32 = benchmark(mat_shape, torch.float32)
# float16
t_f16, m_f16, v_f16 = benchmark(mat_shape, torch.float16)

print("Tempos (ms):")
print(f"float64: {t_f64:.1f}\nfloat32: {t_f32:.1f}\nfloat16: {t_f16:.1f}")

print("\nMemória (GiB):")
print(f"float64: {m_f64:.1f}\nfloat32: {m_f32:.1f}\nfloat16: {m_f16:.1f}")

print("\nMédia dos valores:")
print(f"float64: {v_f64:.3f}\nfloat32: {v_f32:.3f}\nfloat16: {v_f16:.3f}")

Tempos (ms):
float64: 1274.5
float32: 38.2
float16: 11.1

Memória (GiB):
float64: 10.0
float32: 5.0
float16: 2.5

Média dos valores:
float64: 913.142
float32: 914.738
float16: 912.500


Note as diferenças de tempo. Os cálculos demoram muito mais em float64, e a multiplicação em float16 é mais de 3x mais rápida do que em float32! Isso representa um potencial de speedup de mais de 3x somente modificando a precisão! Mas note que os resultados apresentam pequenas diferenças.

O Pytorch possui duas abordagens para realizar cálculos em meia precisão :

In [3]:
# Autocast para float16, em operações selecionadas. Multiplicação matricial
# é uma dessas operações
with torch.autocast(device_type="cuda", dtype=torch.float16):
    t_af16, m_af16, v_af16 = benchmark(mat_shape, torch.float32)

# Uso do formato tf32 (tensorfloat32) para realizar a multiplicação
torch.set_float32_matmul_precision("high")
t_tf32, m_tf32, v_tf32 = benchmark(mat_shape, torch.float32)

print("Tempos (ms):")
print(f"autocast: {t_af16:.1f}\ntensorfloat32: {t_tf32:.1f}")

print("\nMemória (GiB):")
print(f"autocast: {m_af16:.1f}\ntensorfloat32: {m_tf32:.1f}")

print("\nResultados:")
print(f"autocast: {v_af16:.3f}\ntensorfloat32: {v_tf32:.3f}")

Tempos (ms):
autocast: 21.2
tensorfloat32: 22.2

Memória (GiB):
autocast: 7.5
tensorfloat32: 5.0

Resultados:
autocast: 914.500
tensorfloat32: 914.558


O autocast e o formato tf32 permitem um speedup em relação à float32. 

Há também o formato bfloat32 que possui a mesma magnitude que float32 mas menos resolução. As informações sobre os formatos do Pytorch são:

In [4]:
def pformat(format):
    info = torch.finfo(format)
    print(f"{info.max=}, {info.smallest_normal=}, {info.eps=}")

pformat(torch.float32)
pformat(torch.float16)
pformat(torch.bfloat16)

info.max=3.4028234663852886e+38, info.smallest_normal=1.1754943508222875e-38, info.eps=1.1920928955078125e-07
info.max=65504.0, info.smallest_normal=6.103515625e-05, info.eps=0.0009765625
info.max=3.3895313892515355e+38, info.smallest_normal=1.1754943508222875e-38, info.eps=0.0078125


### Biblioteca bitsandbytes

A biblioteca bitsandbytes possibilita quantizar os parâmetros de camadas lineares, o que reduz o custo de memória.

In [147]:
import torch
import torch.nn as nn
from bitsandbytes.nn import Linear4bit

n = 32
# Modelo em float32
model = nn.Linear(n, n)

# Criação de uma camada Linear4bit e cópia dos pesos do modelo
quantized_model = Linear4bit(n, n)
quantized_model.load_state_dict(model.state_dict())
# O modelo é quantizado ao copiar para a GPU:
quantized_model = quantized_model.to("cuda") 

print(model.weight.numel(), model.weight.dtype)
print(quantized_model.weight.numel(), quantized_model.weight.dtype)

1024 torch.float32
512 torch.uint8


O modelo original com 1024 parâmetros passou a ocupar 512\*8 bits. Isso corresponde a 512\*8/1024 = 4 bits/parâmetro.

### Memória *page-locked*

Dados copiados da CPU para a GPU não podem residir em memória pageada. Por padrão, um tensor do Pytorch é alocado em memória pageada. É possível criar um tensor em um intervalo de memória não pageada pelo sistema. Esse processo é denominado de "pinned memory", ou page-locked. Ele aumenta de forma significativa o desempenho da cópia dos dados.

In [6]:
def copy(mat_shape, pin):

    recorder = PerfRecorder()

    x = torch.rand(mat_shape)
    if pin:
        x = x.pin_memory()

    recorder.start()
    x = x.to("cuda")
    t_gpu, _ = recorder.end()
    
    return t_gpu

t_nopin = copy(mat_shape, False)
t_pin = copy(mat_shape, True)

print("Tempos (ms):")
print(f"pin=False: {t_nopin:.1f}\npin=True: {t_pin:.1f}")

Tempos (ms):
pin=False: 213.9
pin=True: 108.8


Copiar tensores com pin_memory habilitado é quase 2x mais rápido. Isso é muito relevante, considerando que copiar dados da CPU para a GPU é uma operação extremamente custosa (note que demora mais para copiar uma matriz do que para realizar a multiplicação matricial entre centenas de milhões de valores).

Os dataloaders do Pytorch possuem uma parâmetro `pin_memory` que quando True utiliza essa técnica.

### Checkpoint de gradiente/ativação

Checkpoint de gradiente é uma técnica que permite utilizar menos memória da GPU em troca de um custo computacional um pouco maior. Uma ou mais camadas do modelo são definidas como *checkpoints*. No processamento direto (forward), a ativação de uma camada checkpoint é salva, e as ativações das camadas seguintes não são salvas no grafo de computação. No momento do cálculo de gradientes, as ativações a partir da camada checkpoint são recalculadas.

In [88]:
from torchvision.models import resnet50


def bench_model(model, bs, n=5, n_warm=2):
    """Mede o tempo de execução e uso de memória de um loop de treinamento."""

    optim = torch.optim.SGD(model.parameters())
    x = torch.rand(bs, 3, 224, 224, device="cuda")
    recorder = PerfRecorder()

    for _ in range(n_warm):
        _ = model(x)

    recorder.start()
    for _ in range(n):
        scores = model(x)
        loss = scores.mean()
        loss.backward()
        optim.step()
    t_gpu, max_memory = recorder.end()

    return t_gpu, max_memory

model = resnet50().to("cuda")

t_resnet, m_resnet = bench_model(model, bs=96)
print("Modelo base:")
print(f"Tempo (ms): {t_resnet:.1f}\nMemória (GiB): {m_resnet:.1f}")


Modelo base:
Tempo (ms): 955.8
Memória (GiB): 8.2


In [89]:
from torch import nn
from torch.utils.checkpoint import checkpoint


class ResNetChkp(nn.Module):
    """Cria um modelo ResNet utilizando a técnica de checkpoint de gradiente."""

    def __init__(self, model):
        super().__init__()

        # Grupos de camadas da ResNet. Os primeiros dois grupos possuem 
        # ativações de alta resolução
        self.group1 = nn.Sequential(
            model.conv1,
            model.bn1,
            model.relu,
            model.maxpool,
            model.layer1[:2]
        )
        self.group2 = nn.Sequential(
            model.layer1[2:],
            model.layer2[:2]
        )
        self.group3 = nn.Sequential(
            model.layer2[2:],
            model.layer3,
            model.layer4,
            model.avgpool    
        )
        self.fc = model.fc

    def forward(self, x):

        # Aplica grupos 1 e 2 utilizando checkpoint. As ativações das camadas não
        # serão salvas no grafo de computação. Quando elas forem necessárias,
        # serão recalculadas através de uma nova aplicação das camadas na entrada
        x = checkpoint(self.group1, x, use_reentrant=False)
        x = checkpoint(self.group2, x, use_reentrant=False)
        x = self.group3(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x
    
resnet_ckp = ResNetChkp(model)

t_resnet, m_resnet = bench_model(resnet_ckp, bs=96)
print("Modelo com checkpoint")
print(f"Tempo (ms): {t_resnet:.1f}\nMemória (GiB): {m_resnet:.1f}")

Modelo com checkpoint
Tempo (ms): 1124.6
Memória (GiB): 3.6


O modelo com checkpoint utiliza 43% da memória do modelo original e possui tempo de processamento 15% maior. 

### LoRA

A técnica LoRA consiste em realizar o refinamento de um modelo através de matrizes que aproximam a alteração que os parâmetros do modelo sofrem durante o refinamento. Para entender a motivação da técnica, veremos primeiro como uma matriz pode ser aproximada através da técnica SVD (Singular Value Decomposition).

In [ ]:
import torch

torch.set_printoptions(sci_mode=False)

def generate_data():
    """Ignore esta função! Ela gera dados para o exmeplo!"""

    dweights = torch.tensor([
        [-1.3296, -0.6248,-0.1404, 0.5190,-0.4966],
        [-9.3562,-10.8182,-9.9593, 5.3702,-5.5752],
        [-0.7968,  1.4863, 3.5823, 0.1940, 0.2743],
        [ 0.0770, -0.5937,-1.1150, 0.0554,-0.1841],
        [ 3.8684,  2.8810, 1.2379,-2.0317, 1.8213]
        ])

    weights = torch.randn(5, 5)
    weights_tunned = weights + dweights

    return weights, weights_tunned

weights, weights_tunned = generate_data()

Suponha que os parâmetros de uma camada do modelo estão contidos na matrix `weights`. Após o refinamento desse modelo, os parâmetros dessa camada passarão a ser `weights_tunned`

In [6]:
print("Original:")
print(weights)
print("\nApós refinamento:")
print(weights_tunned)

delta_weights = weights_tunned - weights
print("\nAlteração sofrida pelos pesos durante o refinamento:")
print(delta_weights)

Original:
tensor([[ 0.4876, -1.4013,  1.5595,  0.8437, -1.6104],
        [-0.1098, -0.6601,  1.1189,  1.2908, -0.2013],
        [-0.2609, -0.3739, -1.0023, -1.1980, -0.3112],
        [-1.6720, -0.7299, -1.2348, -0.3758,  0.4007],
        [-0.4148, -2.1701,  0.1992,  0.6949, -0.8319]])

Após refinamento:
tensor([[ -0.8420,  -2.0261,   1.4191,   1.3627,  -2.1070],
        [ -9.4660, -11.4783,  -8.8404,   6.6610,  -5.7765],
        [ -1.0577,   1.1124,   2.5800,  -1.0040,  -0.0369],
        [ -1.5950,  -1.3236,  -2.3498,  -0.3204,   0.2166],
        [  3.4536,   0.7109,   1.4371,  -1.3368,   0.9894]])

Alteração sofrida pelos pesos durante o refinamento:
tensor([[ -1.3296,  -0.6248,  -0.1404,   0.5190,  -0.4966],
        [ -9.3562, -10.8182,  -9.9593,   5.3702,  -5.5752],
        [ -0.7968,   1.4863,   3.5823,   0.1940,   0.2743],
        [  0.0770,  -0.5937,  -1.1150,   0.0554,  -0.1841],
        [  3.8684,   2.8810,   1.2379,  -2.0317,   1.8213]])


A técnica SVD permite decompor uma matriz em uma base de vetores que proporcionam aproximações cada vez mais precisas do matriz:

In [7]:
def approximate_matrix(U, S, V, rank):
    """Calcula uma aproximação de uma matriz a partir de sua decomposição SVD.
    A aproximação é feita utilizando os `rank` primeiros valores singulares."""

    M = torch.zeros(U.shape[0], V.shape[1])
    for r in range(rank):
        M += S[r] * U[:, [r]] @ V[[r], :]

    return M
    
U, S, V = torch.linalg.svd(delta_weights)
# Aproximação considerando um rank intrínseco de 2:
approximate_matrix(U, S, V, rank=2)

tensor([[ -1.2172,  -0.7274,  -0.0740,   0.6139,  -0.5138],
        [ -9.3642, -10.8091,  -9.9645,   5.3628,  -5.5772],
        [ -0.8251,   1.5132,   3.5653,   0.1698,   0.2768],
        [  0.0620,  -0.5875,  -1.1218,   0.0454,  -0.1684],
        [  3.8786,   2.8764,   1.2426,  -2.0247,   1.8113]])

Isso quer dizer que a matriz pode ser aproximada pelas duas seguintes matrizes:

In [9]:
rank = 2
A = U[:, :rank]*S[:rank]
B = V[:rank]

print(A)
print(B)

tensor([[ -1.3803,   0.8670],
        [-19.0695,  -0.1917],
        [  2.3163,   3.2283],
        [ -0.9421,  -0.8663],
        [  5.3039,  -2.0274]])
tensor([[ 0.4972,  0.5662,  0.5152, -0.2838,  0.2937],
        [-0.6123,  0.0625,  0.7348,  0.2562, -0.1250]])


Portanto, os pesos refinados podem ser escritos como:

In [11]:
weights_tunned_approx = weights + A@B

print("Pesos refinados:")
print(weights_tunned)
print("\nAproximação dos pesos refinados:")
print(weights_tunned_approx)

Pesos refinados:
tensor([[ -0.8420,  -2.0261,   1.4191,   1.3627,  -2.1070],
        [ -9.4660, -11.4783,  -8.8404,   6.6610,  -5.7765],
        [ -1.0577,   1.1124,   2.5800,  -1.0040,  -0.0369],
        [ -1.5950,  -1.3236,  -2.3498,  -0.3204,   0.2166],
        [  3.4536,   0.7109,   1.4371,  -1.3368,   0.9894]])

Aproximação dos pesos refinados:
tensor([[ -0.7296,  -2.1287,   1.4855,   1.4576,  -2.1243],
        [ -9.4740, -11.4692,  -8.8456,   6.6537,  -5.7785],
        [ -1.0860,   1.1392,   2.5630,  -1.0282,  -0.0345],
        [ -1.6100,  -1.3174,  -2.3567,  -0.3304,   0.2323],
        [  3.4637,   0.7063,   1.4418,  -1.3298,   0.9794]])


Ao invés de refinar a matriz `weights` possuindo 25 valores, podemos refinar as matrizes `A` e `B` possuindo 20 elementos. Em geral, ao invés de refinar uma matriz possuindo $n*m$ parâmetros, podemos refinar duas matrizes com um total de $n*r + m*r$ parâmetros. Se $m=n$, temos uma redução de $n/2r$ no número de parâmetros.

Por exemplo, as matrizes das camadas de atenção do modelo Llama 3 70B possuem tamanho $8192\times 8192$. Um valor comum para o rank é 8. Com isso, ao invés de otimizar 67108864 parâmetros, a técnica LoRA permite otimizar apenas $2*8192*8=131072$ parâmetros. Uma redução de $\times 500$ no número de parâmetros. Isso também reduz drasticamente o uso de memória.

Uma outra vantagem do LoRA é que ela permite que modelos refinados em diferentes tarefas sejam rapidamente aplicados. O modelo base é mantido na memória, e apenas os parâmetros extra específicos de uma tarefa são trocados. Isso também permite compartilhar com a comunidade os parâmetros refinados com menor custo. 

### Compilação do grafo de execução

Cada operação realizada pelo Pytorch na GPU envolve a chamada de uma função cuda. Isso representa um custo adicional de comunicação entre CPU e GPU. Adicionalmente, existem diversas operações que podem ser otimizadas. Por exemplo, uma operação de convolução seguida de uma operação batchnorm pode ser representada por uma única operação linear dado que ambas as operações são lineares. Um loop de treinamento envolve chamar exatamente as mesma funções diversas vezes. O Pytorch permite compilar um grafo de executação que otimizará uma sequência de operações realizadas na GPU. Isso é feito através da função `torch.compile()`. Por exemplo, dado um modelo, podemos fazer:

`model = torch.compile(model, fullgraph=True, dynamic=False)`

O modelo pode então ser utilizado no loop de treinamento. Isso pode levar a speedups significativos. Mas há restrições nos tipos de modelos que podem ser compilados.